# NB06 — Hospital-level heterogeneity

**Purpose.** Quantify how calibration and discrimination of the transported model vary across eICU hospitals **beyond chance**: per-hospital O/E, calibration slope and c-statistic; random-effects meta-analysis with 95% prediction intervals; funnel plot with exact Poisson limits; meta-regression on hospital characteristics; comparison with APACHE IVa heterogeneity.

**Research question.** RQ2 — is miscalibration a hospital-level phenomenon?

| | |
|---|---|
| **Inputs** | `models/pred_eicu.parquet`, analysis parquet |
| **Outputs** | Table 3, Figure 3 (funnel + caterpillar), meta-regression |
| **Free-Colab runtime** | 5–10 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB06')

In [ ]:
import statsmodels.api as sm
from scipy.stats import spearmanr
m, e = tu.load_analysis()
pred = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'))
hinfo = e.groupby('hospital_id').agg(numbedscategory=('numbedscategory', 'first'),
                                     teachingstatus=('teachingstatus', 'first'), region=('region', 'first'),
                                     casemix_apache=('apache_pred', 'mean'), mean_age=('age', 'mean')).reset_index()
ph = {}
ph['LightGBM'] = tu.per_hospital(pred, 'p_lgbm')
ph['Logistic regression'] = tu.per_hospital(pred, 'p_lr')
ph['APACHE IVa'] = tu.per_hospital(pred[pred['apache_pred'].notna()], 'apache_pred')
for k, v in ph.items():
    tu.save_table(v.round(4), f"per_hospital_{k.split()[0].lower()}", show=False)
    print(k, ': hospitals', len(v), '| eligible', int(v['eligible'].sum()))

In [ ]:
t3 = pd.concat([tu.summarise_hospitals(v, k) for k, v in ph.items()])
tu.save_table(t3.round(3), 'table3_heterogeneity')

### Funnel plot: how many hospitals are miscalibrated beyond what chance alone would produce?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharey=True)
funnel_rows = []
for ax, k in zip(axes, ['LightGBM', 'APACHE IVa']):
    v = ph[k][ph[k]['expected'] >= 1].copy()
    grid = np.linspace(max(1, v['expected'].min()), v['expected'].max() * 1.05, 300)
    fl = tu.funnel_limits(grid)
    ax.fill_between(fl['expected'], fl['lo_0.998'], fl['hi_0.998'], color=tu.C['grid'], label='99.8% limits')
    ax.plot(fl['expected'], fl['lo_0.95'], color=tu.C['grey'], lw=1, ls='--', label='95% limits')
    ax.plot(fl['expected'], fl['hi_0.95'], color=tu.C['grey'], lw=1, ls='--')
    out95, out998 = tu.outside_funnel(v, 0.95), tu.outside_funnel(v, 0.998)
    ax.scatter(v['expected'], v['oe'], s=14, color=np.where(out998, tu.C['red'], tu.C['blue']),
               edgecolor='white', linewidth=0.4, zorder=3)
    ax.axhline(1, color=tu.C['ink2'], lw=1)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Expected deaths (sum of predicted risk)'); ax.set_ylabel('Observed / expected')
    ax.set_title(f"{k}: {out998.mean()*100:.0f}% of hospitals outside 99.8% limits")
    ax.legend(loc='upper right')
    funnel_rows.append(dict(model=k, hospitals=len(v), pct_outside_95=100 * out95.mean(),
                            expected_by_chance_95=5.0, pct_outside_998=100 * out998.mean(),
                            expected_by_chance_998=0.2,
                            pct_above_998=100 * (out998 & (v['oe'] > 1)).mean(),
                            pct_below_998=100 * (out998 & (v['oe'] < 1)).mean()))
tu.save_fig(fig, 'fig3a_funnel'); plt.show()
funnel = pd.DataFrame(funnel_rows)
tu.save_table(funnel.round(1), 'funnel_summary')

In [ ]:
# Caterpillar plot of per-hospital O/E (eligible hospitals), LightGBM vs APACHE IVa
fig, ax = plt.subplots(figsize=(10, 4))
for i, k in enumerate(['LightGBM', 'APACHE IVa']):
    v = ph[k][ph[k]['eligible']].sort_values('oe').reset_index(drop=True)
    x = np.linspace(0, 1, len(v))
    lo, hi = np.exp(v['ln_oe'] - 1.96 * v['ln_oe_se']), np.exp(v['ln_oe'] + 1.96 * v['ln_oe_se'])
    ax.vlines(x, lo, hi, color=tu.SERIES[i], alpha=0.35, lw=1)
    ax.scatter(x, v['oe'], s=8, color=tu.SERIES[i], label=f'{k} (k={len(v)})', zorder=3)
ax.axhspan(*tu.OE_ACCEPT, color=tu.C['aqua'], alpha=0.08, label='Acceptable band 0.8-1.2')
ax.axhline(1, color=tu.C['ink2'], lw=1); ax.set_yscale('log')
ax.set_xlabel('Hospitals ranked by O/E'); ax.set_ylabel('O/E (95% CI)'); ax.set_xticks([])
ax.legend(loc='upper left'); ax.set_title('Per-hospital calibration-in-the-large')
tu.save_fig(fig, 'fig3b_caterpillar'); plt.show()

### Do the same hospitals look miscalibrated for both models? (hospital effect vs model effect)

In [ ]:
j = ph['LightGBM'][['hospital_id', 'ln_oe', 'eligible']].merge(
    ph['APACHE IVa'][['hospital_id', 'ln_oe']], on='hospital_id', suffixes=('_lgbm', '_apache'))
j = j[j['eligible']].dropna()
rho, pv = spearmanr(j['ln_oe_lgbm'], j['ln_oe_apache'])
print(f'Spearman rho between hospital ln(O/E) of LightGBM and APACHE IVa: {rho:.3f} (p={pv:.2g}, k={len(j)})')
fig, ax = plt.subplots(figsize=(4.2, 4))
ax.scatter(np.exp(j['ln_oe_apache']), np.exp(j['ln_oe_lgbm']), s=12, color=tu.C['blue'])
ax.set_xscale('log'); ax.set_yscale('log'); ax.axhline(1, color=tu.C['grey'], lw=1); ax.axvline(1, color=tu.C['grey'], lw=1)
ax.set_xlabel('APACHE IVa O/E'); ax.set_ylabel('LightGBM O/E'); ax.set_title(f'Hospital O/E agreement (rho={rho:.2f})')
tu.save_fig(fig, 'fig3c_oe_agreement'); plt.show()

### Meta-regression: which hospital characteristics explain LightGBM miscalibration?

In [ ]:
def meta_regression(v, label):
    d = v[v['eligible']].merge(hinfo, on='hospital_id', how='left').dropna(subset=['ln_oe', 'ln_oe_se'])
    tau2 = tu.meta_analysis(d['ln_oe'], d['ln_oe_se'])['tau2']
    d['teaching'] = (d['teachingstatus'].astype(str).str.lower() == 'true').astype(float)
    X = pd.get_dummies(d[['numbedscategory', 'region']].fillna('Unknown'), drop_first=True, dtype=float)
    X['teaching'] = d['teaching']
    X['casemix_apache_x10'] = 10 * d['casemix_apache'].fillna(d['casemix_apache'].mean())
    X['mean_age_per10y'] = d['mean_age'] / 10
    X = sm.add_constant(X)
    fit = sm.WLS(d['ln_oe'], X, weights=1 / (d['ln_oe_se'] ** 2 + tau2)).fit()
    res = pd.DataFrame(dict(term=fit.params.index, coef_ln_oe=fit.params.values, se=fit.bse.values, p=fit.pvalues.values))
    res.insert(0, 'model', label)
    tau2_res = max(0.0, (np.sum((fit.resid ** 2) / d['ln_oe_se'] ** 2) - (len(d) - X.shape[1])) /
                   np.sum(1 / d['ln_oe_se'] ** 2))
    return res, dict(model=label, k=len(d), tau2_total=tau2, tau2_residual_approx=tau2_res)

mr, mr_info = [], []
for k in ['LightGBM', 'APACHE IVa']:
    r, info = meta_regression(ph[k], k); mr.append(r); mr_info.append(info)
mr = pd.concat(mr)
tu.save_table(mr.round(4), 'meta_regression_ln_oe')
print(pd.DataFrame(mr_info))

In [ ]:
tu.finish_notebook({
    'table3': t3.round(4).to_dict('records'),
    'funnel': funnel.round(2).to_dict('records'),
    'spearman_oe_lgbm_vs_apache': dict(rho=round(float(rho), 3), p=float(pv), k=len(j)),
    'meta_regression': mr.round(4).to_dict('records'), 'meta_regression_tau2': mr_info,
})